# Trace Count v5.2: switch-token diagnostics

目标：解释为什么 mixed thinking-toggle Transformer 里，开关看起来学会了，但 targeted retrieval 不显著。

核心检查：

1. `<Think/>` / `</Think>` 的 switch logits 是否把 thinking 与 non-thinking 分开；
2. retrieval 是否测在正确的 **prediction query** 上，而不是 marker token 已经可见之后；
3. marker-only trace 是否因为缺少显式 `index_token_k`，天然不容易形成 v2 那种 k-to-k retrieval head。
            

## 1. Setup

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
INSTALL_DEPS = False

if IN_COLAB:
    repo_dir = Path("/content/Synthetic_CoT_NiaH_Count")
    cwd = Path.cwd()
    if (cwd / ".git").exists() or (cwd / "README.md").exists():
        repo_dir = cwd
    elif not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.40", "pandas", "matplotlib", "tqdm"], check=True)

import pandas as pd
from IPython.display import Markdown, display, Image

display(Markdown(f"**Repo root:** `{ROOT}`"))

## 2. Runtime settings

In [ ]:
# Point this to a completed v5 run. If it does not exist, run Trace_Count_v5_Colab first.
RUN_DIR = Path("outputs/v5")
if not (RUN_DIR / "checkpoints" / "final.pt").exists():
    candidates = (
        list(Path("outputs").glob("v5*/**/final.pt"))
        + list(Path("runs").glob("**/final.pt"))
        + list(Path("colab_results").glob("*v5*/**/final.pt"))
    )
    if candidates:
        RUN_DIR = candidates[-1].parents[1]

EXAMPLES_PER_COUNT = 100
DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"

print({"RUN_DIR": str(RUN_DIR), "EXAMPLES_PER_COUNT": EXAMPLES_PER_COUNT, "DEVICE": DEVICE})
            

## 3. Run v5.2 diagnostics

In [ ]:
from synthetic_counting_extensions.v5_2_switch_diagnostics import run_switch_and_retrieval_diagnostics

outputs = run_switch_and_retrieval_diagnostics(RUN_DIR, examples_per_count=EXAMPLES_PER_COUNT, device=DEVICE)
display(Markdown(f"**Report:** `{RUN_DIR / 'v5_2_switch_diagnostics' / 'report' / 'report.html'}`"))
display(outputs["switch_summary"])
display(outputs["prediction_query_head_summary"].sort_values(["correct_top1", "correct_prompt_needle_mass"], ascending=False).head(12))
            

## 4. Display figures

In [ ]:
FIG_DIR = RUN_DIR / "v5_2_switch_diagnostics" / "figures"
for name in [
    "switch_probability_summary.png",
    "prediction_query_correct_top1.png",
    "prediction_query_correct_mass.png",
    "prediction_query_marker_margin.png",
    "post_marker_correct_top1.png",
]:
    path = FIG_DIR / name
    if path.exists():
        display(Markdown(f"### {name}"))
        display(Image(filename=str(path)))
            

## 5. Save / GitHub / disconnect

In [ ]:
# Save result folder to Google Drive.
SAVE_TO_DRIVE = True
DRIVE_DEST_ROOT = "/content/drive/MyDrive/Colab_Notebooks/CoT_Counting/Synthetic_CoT_NiaH_Count/colab_results"

if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    dest_root = Path(DRIVE_DEST_ROOT)
    dest_root.mkdir(parents=True, exist_ok=True)
    if "RUN_DIR" in globals() and RUN_DIR is not None:
        src = Path(RUN_DIR)
    elif "OUT_ROOT" in globals():
        src = Path(OUT_ROOT)
    else:
        src = None
    if src is not None and src.exists():
        dest = dest_root / src.name
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(src, dest)
        display(Markdown(f"**Saved to Drive:** `{dest}`"))
    else:
        display(Markdown("No RUN_DIR/OUT_ROOT found to save."))
else:
    display(Markdown("Drive save skipped."))

In [ ]:
# Optional: commit and push notebook/code changes to GitHub.
PUSH_TO_GITHUB = False
GIT_COMMIT_MESSAGE = "Add synthetic counting experiment notebook"

if PUSH_TO_GITHUB:
    subprocess.run(["git", "status", "--short"], check=False)
    subprocess.run(["git", "add", "notebooks", "synthetic_counting_extensions", "scripts"], check=True)
    subprocess.run(["git", "commit", "-m", GIT_COMMIT_MESSAGE], check=True)
    subprocess.run(["git", "push"], check=True)
else:
    display(Markdown("GitHub push skipped. Set `PUSH_TO_GITHUB = True` after checking the diff."))

In [ ]:
# Optional: disconnect Colab runtime after saving.
AUTO_DISCONNECT_AFTER_SAVE = False

if AUTO_DISCONNECT_AFTER_SAVE and IN_COLAB:
    from google.colab import runtime
    runtime.unassign()
else:
    display(Markdown("Auto-disconnect skipped."))